# 🛰️ Orbital Insight Analytics — Exploratory Data Analysis

This notebook demonstrates the full data pipeline:
1. Simulating satellite imagery (NIR + Red → NDVI)
2. Exploring deforestation, agriculture, and urban datasets
3. Training and evaluating ML models
4. Visualising results with matplotlib and plotly

All data is synthetically generated to demonstrate the platform.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from utils.ndvi import simulate_satellite_bands, calculate_ndvi, ndvi_statistics, classify_ndvi
from utils.data_processing import (
    extract_ndvi_features, compute_change_metrics,
    dataframe_from_ndvi_timeseries
)
from utils.visualization import (
    ndvi_to_base64, deforestation_map_to_base64,
    urban_growth_map_to_base64, productivity_trend_chart,
    ndvi_histogram, feature_importance_chart, NDVI_CMAP
)
from models import deforestation_model, agriculture_model, urban_model
from data.simulate_data import (
    simulate_deforestation_series,
    simulate_agriculture_dataset,
    simulate_urban_growth_dataset,
)

print('All imports successful ✓')

---
## 1. Satellite Imagery Simulation

In [ ]:
# Simulate NIR and Red bands for a healthy forest area
healthy_bands = simulate_satellite_bands(height=64, width=64, vegetation_fraction=0.80, seed=42)
deforested_bands = simulate_satellite_bands(height=64, width=64, vegetation_fraction=0.20, seed=99)

ndvi_healthy = healthy_bands['ndvi']
ndvi_deforested = deforested_bands['ndvi']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title, cmap in zip(
    axes,
    [healthy_bands['nir'], healthy_bands['red'], ndvi_healthy],
    ['NIR Band', 'Red Band', 'NDVI'],
    ['gray', 'Reds', NDVI_CMAP]
):
    im = ax.imshow(data, cmap=cmap, vmin=-1 if 'NDVI' in title else 0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
fig.suptitle('Simulated Satellite Bands — Healthy Forest (80% vegetation)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('NDVI Statistics (healthy):')
for k, v in ndvi_statistics(ndvi_healthy).items():
    print(f'  {k}: {v:.4f}')

---
## 2. Deforestation Analysis

In [ ]:
# Load simulated deforestation time series
df_def = simulate_deforestation_series(n_regions=50, n_years=10, seed=0)
print(df_def.head(10))
print(f'\nShape: {df_def.shape}')
print(f'Deforested fraction: {df_def.deforested.mean():.2%}')

In [ ]:
# Plot NDVI trend for a few regions
fig, ax = plt.subplots(figsize=(12, 5))
for region_id in [0, 5, 10, 15, 20]:
    region_data = df_def[df_def['region_id'] == region_id]
    ax.plot(region_data['year'], region_data['ndvi_mean'],
            marker='o', label=f'Region {region_id}')

ax.axhline(0.35, color='red', linestyle='--', alpha=0.7, label='Deforestation threshold (0.35)')
ax.set_xlabel('Year')
ax.set_ylabel('Mean NDVI')
ax.set_title('NDVI Trend by Region (Deforestation Time Series)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Before / After NDVI comparison
change = compute_change_metrics(ndvi_healthy, ndvi_deforested)
print('Change metrics:')
for k, v in change.items():
    print(f'  {k}: {v:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
diff = ndvi_healthy - ndvi_deforested
for ax, data, title, cmap, vrange in zip(
    axes,
    [ndvi_healthy, ndvi_deforested, diff],
    ['Before (Healthy)', 'After (Deforested)', 'Δ NDVI (Loss)'],
    [NDVI_CMAP, NDVI_CMAP, 'RdYlGn_r'],
    [(-1, 1), (-1, 1), (-1, 1)]
):
    im = ax.imshow(data, cmap=cmap, vmin=vrange[0], vmax=vrange[1])
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('Deforestation Detection: Before vs After', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Train and evaluate deforestation classifier
metrics_def = deforestation_model.train(n_samples=2000, seed=42)
print(f'Deforestation Classifier Accuracy: {metrics_def["accuracy"]:.4f}')
print('Feature Importances:')
for name, imp in sorted(
    zip(metrics_def['feature_names'], metrics_def['feature_importances']),
    key=lambda x: -x[1]
):
    print(f'  {name}: {imp:.4f}')

---
## 3. Agricultural Productivity

In [ ]:
df_agr = simulate_agriculture_dataset(n_samples=1000, seed=1)
print(df_agr.describe())
print(f'\nCrop distribution:\n{df_agr.crop_type.value_counts()}')

In [ ]:
import plotly.express as px

fig = px.scatter(
    df_agr, x='ndvi_mean', y='yield_tons_ha',
    color='crop_type', size='soil_moisture',
    labels={'ndvi_mean': 'Mean NDVI', 'yield_tons_ha': 'Yield (tons/ha)'},
    title='Crop Yield vs NDVI by Crop Type',
    template='plotly_white'
)
fig.show()

In [ ]:
# Train agriculture model
metrics_agr = agriculture_model.train(n_samples=3000, seed=42)
print(f'RMSE: {metrics_agr["rmse"]:.4f} tons/ha')
print(f'MAE:  {metrics_agr["mae"]:.4f} tons/ha')
print(f'R²:   {metrics_agr["r2"]:.4f}')

# Predict for a sample field
sample = {
    'ndvi_mean': 0.72, 'ndvi_p50': 0.71,
    'soil_moisture': 0.65, 'temperature_avg': 26.0,
    'precipitation_mm': 550.0, 'crop_type': 'soy',
    'days_to_harvest': 110.0
}
pred = agriculture_model.predict(sample)
print(f'\nSample prediction: {pred}')

---
## 4. Urban Growth Monitoring

In [ ]:
df_urb = simulate_urban_growth_dataset(n_cities=30, seed=2)
print(df_urb.head(10))

# Plot urban area growth for a few cities
fig, ax = plt.subplots(figsize=(12, 5))
for city_id in [0, 5, 10, 15, 20]:
    city_data = df_urb[df_urb['city_id'] == city_id]
    ax.plot(city_data['year'], city_data['urban_area_km2'],
            marker='o', label=f'City {city_id}')
ax.set_xlabel('Year')
ax.set_ylabel('Urban Area (km²)')
ax.set_title('Urban Expansion by City (2015–2024)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Train urban model
metrics_urb = urban_model.train(n_samples=4000, seed=42)
print(f'Urban Classifier Accuracy: {metrics_urb["accuracy"]:.4f}')

# Simulate urban growth detection
before_bands = simulate_satellite_bands(64, 64, vegetation_fraction=0.85, seed=10)
after_bands = simulate_satellite_bands(64, 64, vegetation_fraction=0.60, seed=11)

before_map = urban_model.predict_map(
    before_bands['ndvi'], before_bands['red'], before_bands['nir']
)
after_map = urban_model.predict_map(
    after_bands['ndvi'], after_bands['red'], after_bands['nir']
)

stats = urban_model.compute_urban_growth_stats(before_map, after_map)
print('\nUrban growth statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# Visualise urban classification maps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
new_urban = ((after_map == 1) & (before_map == 0)).astype(int)
for ax, data, title, cmap in zip(
    axes,
    [before_map, after_map, new_urban],
    ['Before (Urban)', 'After (Urban)', 'New Urban Areas'],
    ['Blues', 'Blues', 'Reds']
):
    im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.axis('off')
fig.suptitle('Urban Growth Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Model Performance Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': 'Deforestation Classifier (Random Forest)', 'Metric': 'Accuracy', 'Value': f"{metrics_def['accuracy']:.4f}"},
    {'Model': 'Agriculture Regressor (Gradient Boosting)', 'Metric': 'RMSE (tons/ha)', 'Value': f"{metrics_agr['rmse']:.4f}"},
    {'Model': 'Agriculture Regressor (Gradient Boosting)', 'Metric': 'R²', 'Value': f"{metrics_agr['r2']:.4f}"},
    {'Model': 'Urban Classifier (Random Forest)', 'Metric': 'Accuracy', 'Value': f"{metrics_urb['accuracy']:.4f}"},
])
print(summary.to_string(index=False))